In [ ]:
import glob
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import root_mean_squared_error

In [ ]:
input_files = sorted(glob.glob("train/input_2023_w*.csv"))
output_files = sorted(glob.glob("train/output_2023_w*.csv"))

df_input = pd.concat([pd.read_csv(f) for f in input_files], ignore_index=True)
df_output = pd.concat([pd.read_csv(f) for f in output_files], ignore_index=True)

print("Input shape:", df_input.shape)
print("Output shape:", df_output.shape)

# Make sure player_to_predict is int
df_input["player_to_predict"] = df_input["player_to_predict"].astype(int)

# Engineer Features

In [ ]:
def parse_height_to_inches(h):
    if pd.isna(h):
        return np.nan
    try:
        ft, inch = str(h).split("-")
        return int(ft) * 12 + int(inch)
    except Exception:
        return np.nan

In [ ]:
df_input["height_in"] = df_input["player_height"].apply(parse_height_to_inches)

df_input["bmi"] = df_input["player_weight"] / (df_input["height_in"] ** 2)

df_input["age"] = 2026 - pd.to_datetime(df_input["player_birth_date"]).dt.year

# Assume x in [0, 120], y in [0, 53.3]
# For plays moving left, flip coordinates.
is_left = df_input["play_direction"].str.lower() == "left"

# normalized x/y (offense always left->right)
df_input["x_std"] = df_input["x"]
df_input["y_std"] = df_input["y"]
df_input.loc[is_left, "x_std"] = 120.0 - df_input.loc[is_left, "x"]
df_input.loc[is_left, "y_std"] = 53.3 - df_input.loc[is_left, "y"]

# normalize ball landing spot the same way
df_input["ball_land_x_std"] = df_input["ball_land_x"]
df_input["ball_land_y_std"] = df_input["ball_land_y"]
df_input.loc[is_left, "ball_land_x_std"] = 120.0 - df_input.loc[is_left, "ball_land_x"]
df_input.loc[is_left, "ball_land_y_std"] = 53.3 - df_input.loc[is_left, "ball_land_y"]

# simple numeric encoding for play_direction if needed
df_input["play_dir_num"] = np.where(is_left, -1, 1)

df_input["is_offense"] = (df_input["player_side"] == "Offense").astype(int)
df_input["is_defense"] = (df_input["player_side"] == "Defense").astype(int)

df_input["is_targeted"] = (df_input["player_role"] == "Targeted Receiver").astype(int)
df_input["is_passer"] = (df_input["player_role"] == "Passer").astype(int)
df_input["is_route_runner"] = df_input["player_role"].isin(
    ["Targeted Receiver", "Other Route Runner"]
).astype(int)
df_input["is_def_coverage"] = (df_input["player_role"] == "Defensive Coverage").astype(int)

In [ ]:
# Distance from player to ball landing point (using normalized coords)
dx = df_input["ball_land_x_std"] - df_input["x_std"]
dy = df_input["ball_land_y_std"] - df_input["y_std"]
df_input["dist_to_land"] = np.sqrt(dx**2 + dy**2)

# Orientation / direction in radians
o_rad = np.deg2rad(df_input["o"])
dir_rad = np.deg2rad(df_input["dir"])

# Angle from player to ball
angle_to_ball = np.arctan2(dy, dx)
df_input["angle_to_ball"] = angle_to_ball

# angle difference between where player is facing (o) and ball
def normalize_angle(angle):
    # map angle to [-pi, pi]
    return (angle + np.pi) % (2 * np.pi) - np.pi

df_input["angle_diff_oball"] = normalize_angle(angle_to_ball - o_rad)

In [ ]:
# Velocity components from speed and direction of motion
df_input["vel_x"] = df_input["s"] * np.cos(dir_rad)
df_input["vel_y"] = df_input["s"] * np.sin(dir_rad)

# Vector from player to ball (normalized)
to_ball_norm = np.sqrt(dx**2 + dy**2)
# Avoid division by zero
to_ball_norm_safe = to_ball_norm.replace(0, np.nan)

df_input["to_ball_x_unit"] = dx / to_ball_norm_safe
df_input["to_ball_y_unit"] = dy / to_ball_norm_safe

# Cosine similarity between velocity vector and vector to ball
vel_norm = np.sqrt(df_input["vel_x"]**2 + df_input["vel_y"]**2)
vel_norm_safe = vel_norm.replace(0, np.nan)

dot_prod = df_input["vel_x"] * df_input["to_ball_x_unit"] + df_input["vel_y"] * df_input["to_ball_y_unit"]
df_input["movement_alignment"] = dot_prod / (vel_norm_safe)

# Where norms were zero, set alignment to 0 (no movement)
df_input["movement_alignment"] = df_input["movement_alignment"].fillna(0.0)

# Longitudinal / lateral speed (relative to field direction)
df_input["speed_long"] = df_input["s"] * np.cos(dir_rad)  # along x_std axis
df_input["speed_lat"] = df_input["s"] * np.sin(dir_rad)   # along y_std axis

In [ ]:
# In/near red zone: close to opponent end zone
df_input["is_red_zone"] = (df_input["absolute_yardline_number"] <= 20).astype(int)

# Backed up near own end zone
df_input["is_backed_up"] = (df_input["absolute_yardline_number"] >= 80).astype(int)

# Format Taining Data

In [ ]:
id_cols = ["game_id", "play_id", "nfl_id"]

In [ ]:
df_last_in = (
    df_input
    .sort_values("frame_id")
    .groupby(id_cols, as_index=False)
    .tail(1)  # last frame before/at throw for that player
)

In [ ]:
df_last_in = df_last_in[df_last_in["player_to_predict"] == 1].copy()

In [ ]:
cols_for_snapshot = id_cols + [
    "player_to_predict",
    "play_direction",
    "num_frames_output",
    # engineered features you want as inputs:
    "x_std", "y_std",
    "ball_land_x_std", "ball_land_y_std",
    "s", "a", "o", "dir",
    "vel_x", "vel_y",
    "speed_long", "speed_lat",
    "dist_to_land",
    "angle_to_ball",
    "angle_diff_oball",
    "movement_alignment",
    "height_in", "player_weight", "bmi", "age_years",
    "field_pos_scaled", "is_red_zone", "is_backed_up",
    "is_offense", "is_defense",
    "is_targeted", "is_route_runner", "is_passer", "is_def_coverage"
]

In [ ]:
cols_for_snapshot = [c for c in cols_for_snapshot if c in df_last_in.columns]

df_last_in = df_last_in[cols_for_snapshot]

In [ ]:
df_train = df_output.merge(df_last_in, on=id_cols, how="inner")
df_train = df_train.reset_index(drop=True)

In [ ]:
is_left = df_train["play_direction"].str.lower() == "left"

df_train["x_out_std"] = df_train["x"]
df_train["y_out_std"] = df_train["y"]

df_train.loc[is_left, "x_out_std"] = 120.0 - df_train.loc[is_left, "x"]
df_train.loc[is_left, "y_out_std"] = 53.3  - df_train.loc[is_left, "y"]

In [ ]:
df_train["t_norm"] = df_train["frame_id"] / df_train["num_frames_output"]

In [ ]:
df_train = df_train[df_train["player_to_predict"] == 1].copy()

# Drop any rows with missing targets
df_train = df_train.dropna(subset=["x_out_std", "y_out_std"])

print("Train rows for player_to_predict=1:", df_train.shape[0])

In [ ]:
target_cols = ["x_out_std", "y_out_std"]

# Columns we definitely don't want as features
cols_to_exclude = set(id_cols + [
    "frame_id",         # this is output frame index; we use t_norm instead
    "x", "y",           # raw output coords (we use standardized targets)
    "play_direction",   # encoded via standardized coords
    "num_frames_output" # we used it to build t_norm
])

# Also exclude target columns themselves
cols_to_exclude.update(target_cols)

feature_cols = [c for c in df_train.columns if c not in cols_to_exclude]

X = df_train[feature_cols].values
y = df_train[target_cols].values

print(f"Using {len(feature_cols)} feature columns:", feature_cols)

In [ ]:
groups = df_train["game_id"].values  # group by game

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=17)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print("Train size:", X_train.shape[0], "Test size:", X_test.shape[0])

In [ ]:
base_reg = HistGradientBoostingRegressor(
    max_depth=6,
    learning_rate=0.05,
    max_iter=300,
    min_samples_leaf=30,
    random_state=17
)

model = MultiOutputRegressor(base_reg)
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

rmse_x = root_mean_squared_error(y_test[:, 0], y_pred[:, 0])
rmse_y = root_mean_squared_error(y_test[:, 1], y_pred[:, 1])

# Euclidean error per frame (how far off in yards)
euclid_err = np.linalg.norm(y_test - y_pred, axis=1)
euclid_rmse = np.sqrt(np.mean(euclid_err**2))

print(f"RMSE x_out_std: {rmse_x:.4f}")
print(f"RMSE y_out_std: {rmse_y:.4f}")
print(f"RMSE Euclidean (yards): {euclid_rmse:.4f}")